In [28]:
import numpy as np
import pandas as pd

from SymbolicDSGE import Shock, parse_config, solve_model, LinearKalman
from SymbolicDSGE.monte_carlo import (
    MCPipeline,
    custom_transform,
)
from SymbolicDSGE.monte_carlo.step_factories import (
    filter_step,
    simulation_step,
    standardize_step,
    transform_step,
    wald_test_step,
)

In [29]:
ss_seed = np.zeros(5, dtype=np.float64)

model = parse_config("../../MODELS/POST82.yaml")
reference = solve_model(model, ss_seed=ss_seed)

dgp_model = model.with_parameters({"psi_x": 0.2, "psi_pi": 3.0})
dgp = solve_model(dgp_model, ss_seed=ss_seed)

In [30]:
T = 200
n_obs = reference.compiled.n_obs

z = np.zeros((T,), dtype=np.float64)


@custom_transform
def custom_standardize(sample, output) -> int:
    n, p = sample.shape
    for j in range(p):
        mean = 0.0
        for i in range(n):
            mean += sample[i, j]
        mean /= n

        var = 0.0
        for i in range(n):
            var += (sample[i, j] - mean) ** 2
        std = (var / n) ** 0.5

        for i in range(n):
            output[i, j] = (sample[i, j] - mean) / std

    return 0


pipeline = MCPipeline(
    [
        simulation_step(
            T=T,
            target="dgp",
            shocks=[
                Shock(dist="norm", seed=0).joint("e_g", "e_z"),
                Shock(dist="norm", seed=1).joint("e_r"),
            ],
            observables=True,
        ),
        filter_step(
            target="reference",
            obs_source="datagen",
            obs_field="observables",
            filter=LinearKalman(),
            n_retain=0,  # Filter outputs need a lot of memory
        ),
        transform_step(
            "custom_std",
            custom_standardize,
            source="filter",
            field="innov",
            output_shape=(T, n_obs),
        ),
        standardize_step(
            "builtin_std",
            source="filter",
            field="innov",
        ),
        wald_test_step(
            "std_innov_mean",
            source="filter",
            field="std_innov",
            burn_in=20,
            target=np.zeros(n_obs),
            kind="mean",
        ),
    ]
)

In [31]:
mc = pipeline.run(
    models={"reference": reference, "dgp": dgp},
    n_rep=100_000,
    n_jobs=-1,
    verbosity=2,
    fail_fast=False,
)

MC run concluded successfully in 1.57s with 63565.62 it/s.
Per-step Report:

	datagen: 0 failures, 41928.09 worker it/s (2.39 worker-s), 63565.62 wall it/s.
	filter: 0 failures, 4344.60 worker it/s (23.02 worker-s), 63565.62 wall it/s.
	custom_std: 0 failures, 218540.87 worker it/s (0.46 worker-s), 63565.62 wall it/s.
	builtin_std: 0 failures, 217805.80 worker it/s (0.46 worker-s), 63565.62 wall it/s.
	std_innov_mean: 0 failures, 81054.74 worker it/s (1.23 worker-s), 63565.62 wall it/s.


In [32]:
std_innov_mean_res = mc.test_summaries["std_innov_mean"]
print(f"Mean statistic: {std_innov_mean_res.mean_statistic:.3f}")
print(f"Mean p-value: {std_innov_mean_res.mean_pval:.3f}")

Mean statistic: 4.758
Mean p-value: 0.372


In [33]:
t_summary = pd.DataFrame(
    {
        name: {
            "mean_statistic": res.mean_statistic,
            "mean_pval": res.mean_pval,
            "rejection_rate": res.rejection_rate,
            "ci_low": res.pval_confidence_interval()[0],
            "ci_high": res.pval_confidence_interval()[1],
        }
        for name, res in mc.test_summaries.items()
    }
).T
print(t_summary.round(3))

                mean_statistic  mean_pval  rejection_rate  ci_low  ci_high
std_innov_mean           4.758      0.372           0.175    0.37    0.374
